In [6]:
import sys
import os
from pathlib import Path
import yaml

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.core.parser import HiveScriptParser
from src.transformers.optimized_pyspark_transformer import OptimizedPySparkTransformer
from src.jinja.environment import render_template

In [9]:
# Test with the k2_bank DDL file
# script_name = "raw_k2_bank"
script_name = "raw_general_reference_lookup"
# script_name = "raw_mhbos_t_contract"

In [11]:

sql_file_path = project_root / "samples" / "input" / "dml" / "raw" / f"{script_name}.sql"
output_file_path = project_root / "samples" / "converted" / "sparksql_advanced" / f"{script_name}.py"

yaml_file_path = project_root / "configs" / "rules" / "variable.yaml"
with open(yaml_file_path, 'r', encoding='utf-8') as f:
    yaml_config = yaml.safe_load(f)

variable_mapping = {}
for key, val in yaml_config.items():
    if isinstance(val, dict) and 'pyspark' in val:
        variable_mapping[key] = val['pyspark']

print(f"[1] Parsing file {sql_file_path.name}...")
context = HiveScriptParser.parse_file(str(sql_file_path))

print("[2] Initializing Optimized Transformer...")
transformer = OptimizedPySparkTransformer(variable_mapping, config_root=project_root / "configs")
render_model = transformer.transform(context)

print("[3] Rendering Template (optimized_pyspark.jinja)...")
final_script = render_template(
    template_name="pyspark/optimized_pyspark.jinja",
    render_model=render_model,
    template_dir="template"
)

# 2. Load mapping configuration from YAML
output_file_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_file_path, 'w', encoding='utf-8') as f:
    f.write(final_script)

print("\n" + "=" * 50)
print("OPTIMIZED PYTHON FILE RESULT")
print("=" * 50 + "\n")
print(final_script)


[1] Parsing file raw_general_reference_lookup.sql...
[2] Initializing Optimized Transformer...
Optimize for <class 'sqlglot.expressions.Drop'>
Optimize for <class 'sqlglot.expressions.Create'>
Optimize for <class 'sqlglot.expressions.Alter'>
Optimize for <class 'sqlglot.expressions.Insert'>
[3] Rendering Template (optimized_pyspark.jinja)...

OPTIMIZED PYTHON FILE RESULT

"""
Purpose:    RAW-DML, Load the data file into the target table's same-day partition with support for multiple sources
Author:     marco
ModifiedBy: [Your Name]
Usage:      python $ETL_HOME/script/main.py yyyymmdd raw_general_reference_lookup
CreateDate: 20240529
FileType:   DML
Logs:
1. For Hive 3.x on CDP 7.1.5
2. Modified to handle multiple sources and store the source_name column
0.1 set parameter
"""

import os
import sys
sys.path.append("/mapr/Edfdev.kenanga.local/EDF/py_script")

from etl_common_function import run_etl, set_parameter, drop_partition_day, batch_start, batch_end
from pyspark.sql.functions impor

In [ ]:

rule_file_path = project_root / "configs" / "rules" / "optimizations" / "k2.yaml"

with open(rule_file_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)



# for rule in config["rules"]:
#     if not rule.get("enabled", False):
#         continue
#
#     if self._is_rule_triggered(rule, context):
#         action_type = rule.get("action", {}).get("type")
#         handler = action_handlers.get(action_type)
#
#         if handler:
#             # A rule was triggered and a handler exists, stop processing more rules.
#             return handler(rule, node, context)
#
#         # Stop at the first triggered rule, even if the action is unknown.
#         return None

for rule_name, rule_param in config["rules"].items():
    print(rule_param.get("enabled", False))